# Warfarin QML — Main Benchmark k=4 | VQC Seed Robustness

**Mục đích:** Kiểm tra mức biến thiên VQC theo hai seed bổ sung **7 và 123**, ngoài primary seed 42 đã chạy. **Không chọn seed có Macro-F1 tốt nhất** để thay thế primary seed.

**Minh bạch phương pháp:** Hai seed này được bổ sung sau khi đã quan sát kết quả seed 42; đánh dấu `EXPLORATORY_POST_BASELINE` và không gọi là predeclared. Nếu nhóm có bộ seed đã khóa chính thức, phải thống nhất trước khi chạy notebook này và ghi nhận thay đổi protocol.

**Giữ cố định:** IWPC eligible 5410, quantum development pool 600, năm folds 480/120, 10 approved features; per-fold preprocessing → PCA k=4 → MinMax [0,π] clip, `zz_feature_map` reps=2 linear, `real_amplitudes` reps=1 linear, `COBYLA(maxiter=60)`, `StatevectorSampler` 1024 shots, `interpret(bitstring)=bitstring%3`. Seed thay đổi `initial_point` và seed của sampler.

**Không đụng Final Test.** Không tạo Python script trong `scripts/`. Kết quả từng seed/fold lưu JSON riêng trong `docs/results/`, có resume và không ghi đè. Chỉ chạy tới **Cell 4 — seed 7 Fold 0** để kiểm tra trước khi triển khai đầy đủ.

## Cell 1 — Imports, kiểm tra môi trường và các artifacts đã có

In [1]:
from pathlib import Path
from time import perf_counter
from importlib.metadata import version
import sys
import hashlib
import json
import numpy as np
import pandas as pd
import yaml

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, recall_score, confusion_matrix
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA


def find_root(start):
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "configs/feature_policy.yaml").is_file() and (p / "data/splits/quantum_dev_folds.csv").is_file():
            return p
    raise FileNotFoundError("Open notebook inside Warfarin project")

ROOT = find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.prepare_stage4 import load_and_validate_dataset, verify_raw_file
from src.preprocessing import build_preprocessor

OUT = ROOT / "docs/results"
SPLIT_DIR = ROOT / "data/splits"
CLASSES = ["Low", "Intermediate", "High"]
ROBUSTNESS_SEEDS = (7, 123)
PRIMARY_SEED = 42
MAXITER = 60
SHOTS = 1024

assert len(set(ROBUSTNESS_SEEDS)) == 2
assert PRIMARY_SEED not in ROBUSTNESS_SEEDS
assert (OUT / "vqc_k4_5fold_metrics.csv").is_file()
assert all((OUT / f"vqc_k4_fold{i}_diagnostic.json").is_file() for i in range(5))
print("[PASS] Environment:", sys.executable)
print("[PASS] Existing primary VQC seed 42 files found")
print("[INFO] Additional seeds:", ROBUSTNESS_SEEDS)
print("[INFO] Status: exploratory post-baseline; Final Test locked")

[PASS] Environment: c:\Users\admin\anaconda3\envs\nt409\python.exe
[PASS] Existing primary VQC seed 42 files found
[INFO] Additional seeds: (7, 123)
[INFO] Status: exploratory post-baseline; Final Test locked


## Cell 2 — Kiểm tra dữ liệu, hashes, split và primary config

In [2]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

verify_raw_file()
with (ROOT / "configs/feature_policy.yaml").open(encoding="utf-8") as f:
    policy = yaml.safe_load(f)
assert policy["training_allowed"] is True
approved = list(policy["approved_X_columns"])
assert len(approved) == len(set(approved)) == 10

manifest_path = SPLIT_DIR / "split_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["approved_features"] == approved
for filename, hash_expected in manifest["files"].items():
    assert sha256_file(SPLIT_DIR / filename) == hash_expected, filename

X, y = load_and_validate_dataset(approved)
labels = pd.read_csv(ROOT / "data/interim/labels.csv")
folds = pd.read_csv(SPLIT_DIR / "quantum_dev_folds.csv")
splits = pd.read_csv(SPLIT_DIR / "final_split.csv")
y_array = np.asarray(y, dtype=str)
assert len(X) == len(y_array) == len(labels) == 5410
assert np.array_equal(y_array, labels["target_group"].astype(str).to_numpy())
assert labels["raw_row_index"].is_unique and folds["raw_row_index"].is_unique
assert set(folds["fold_id"]) == set(range(5))
qdev_ids = set(folds["raw_row_index"].astype(int))
test_ids = set(splits.loc[splits.partition == "final_test", "raw_row_index"].astype(int))
dev_ids = set(splits.loc[splits.partition == "development", "raw_row_index"].astype(int))
assert len(qdev_ids) == 600 and qdev_ids.issubset(dev_ids)
assert qdev_ids.isdisjoint(test_ids) and dev_ids.isdisjoint(test_ids)
assert len(dev_ids) == 4328 and len(test_ids) == 1082

for fid in range(5):
    baseline = json.loads((OUT / f"vqc_k4_fold{fid}_diagnostic.json").read_text(encoding="utf-8"))
    assert baseline["fold"] == fid and baseline["seed"] == PRIMARY_SEED
    assert baseline["train_n"] == 480 and baseline["val_n"] == 120
    assert baseline["pca_k"] == 4 and baseline["maxiter"] == MAXITER
    assert baseline["shots"] == SHOTS and baseline["optimizer"] == "COBYLA"
    assert baseline["final_test_used"] is False
    assert len(baseline["predictions"]) == 120
    expected = set(folds.loc[folds.fold_id == fid, "raw_row_index"].astype(int))
    actual = {int(r["raw_row_index"]) for r in baseline["predictions"]}
    assert actual == expected

print("[PASS] Raw checksum and locked split hashes")
print("[PASS] All five primary VQC folds verified")
print("[PASS] Final Test excluded")

[PASS] Raw checksum and locked split hashes
[PASS] All five primary VQC folds verified
[PASS] Final Test excluded


## Cell 3 — Hàm chạy VQC cho một seed/fold (dùng lại cho mọi fold)

Lưu sau **từng** fold. Nếu file tồn tại: xác thực cấu hình và IDs rồi bỏ qua, không huấn luyện lại. Có thể chạy tiếp notebook sau khi gián đoạn. Hàm không sử dụng validation trong fit/optimizer.

In [3]:
def run_seed_fold(seed, fold_id):
    seed = int(seed)
    fold_id = int(fold_id)
    assert seed in ROBUSTNESS_SEEDS and fold_id in range(5)
    out_path = OUT / f"vqc_k4_robust_seed{seed}_fold{fold_id}.json"

    val_ids = set(folds.loc[folds.fold_id == fold_id, "raw_row_index"].astype(int))
    train_ids = qdev_ids - val_ids
    assert len(train_ids) == 480 and len(val_ids) == 120
    assert train_ids.isdisjoint(val_ids) and (train_ids | val_ids).isdisjoint(test_ids)
    if out_path.exists():
        old = json.loads(out_path.read_text(encoding="utf-8"))
        assert old["seed"] == seed and old["fold"] == fold_id
        assert old["experiment_status"] == "EXPLORATORY_POST_BASELINE"
        assert old["maxiter"] == MAXITER and old["shots"] == SHOTS
        assert old["pca_k"] == 4 and old["final_test_used"] is False
        assert old["train_n"] == 480 and old["val_n"] == 120
        assert old["feature_policy_sha256"] == sha256_file(ROOT / "configs/feature_policy.yaml")
        assert old["split_manifest_sha256"] == sha256_file(manifest_path)
        assert len(old["predictions"]) == 120 and len(old["loss_history"]) == MAXITER
        saved_ids = {int(p["raw_row_index"]) for p in old["predictions"]}
        assert saved_ids == val_ids
        print(f"[SKIP] Existing verified seed={seed}, fold={fold_id}", flush=True)
        return old

    train_pos = np.flatnonzero(labels["raw_row_index"].isin(train_ids).to_numpy())
    val_pos = np.flatnonzero(labels["raw_row_index"].isin(val_ids).to_numpy())
    X_train, X_val = X.iloc[train_pos], X.iloc[val_pos]
    y_train, y_val = y_array[train_pos], y_array[val_pos]
    assert len(y_train) == 480 and len(y_val) == 120

    preprocessor = build_preprocessor(approved)
    encoded_train = preprocessor.fit_transform(X_train)
    encoded_val = preprocessor.transform(X_val)
    pca = PCA(n_components=4, svd_solver="full")
    pca_train = pca.fit_transform(encoded_train)
    pca_val = pca.transform(encoded_val)
    scaler = MinMaxScaler(feature_range=(0., float(np.pi)))
    z_train = np.clip(scaler.fit_transform(pca_train), 0., np.pi)
    z_val = np.clip(scaler.transform(pca_val), 0., np.pi)
    assert z_train.shape == (480, 4) and z_val.shape == (120, 4)
    assert np.isfinite(z_train).all() and np.isfinite(z_val).all()

    fm = zz_feature_map(feature_dimension=4, reps=2, entanglement="linear")
    ansatz = real_amplitudes(num_qubits=4, reps=1, entanglement="linear")
    assert ansatz.num_parameters == 8
    initial_point = np.random.default_rng(seed).uniform(-np.pi, np.pi, size=ansatz.num_parameters)
    history = []

    def callback(weights, objective):
        history.append(float(objective))
        if len(history) % 10 == 0:
            print(f"seed={seed} fold={fold_id} eval={len(history)} loss={objective:.6f}", flush=True)

    model = VQC(
        feature_map=fm,
        ansatz=ansatz,
        optimizer=COBYLA(maxiter=MAXITER),
        sampler=StatevectorSampler(default_shots=SHOTS, seed=seed),
        loss="cross_entropy",
        interpret=lambda bitstring: bitstring % 3,
        output_shape=3,
        initial_point=initial_point,
        callback=callback,
    )
    print(f"\n=== VQC robustness seed={seed} fold={fold_id} ===", flush=True)
    tic = perf_counter()
    model.fit(z_train, y_train)
    fit_seconds = perf_counter() - tic
    tic = perf_counter()
    pred = np.asarray(model.predict(z_val)).reshape(-1)
    predict_seconds = perf_counter() - tic
    assert len(pred) == 120 and set(pred).issubset(set(CLASSES))
    assert len(history) == MAXITER and np.isfinite(history).all()
    recalls = recall_score(y_val, pred, labels=CLASSES, average=None, zero_division=0)
    cm = confusion_matrix(y_val, pred, labels=CLASSES)
    result = {
        "experiment": "VQC_k4_seed_robustness", "experiment_status": "EXPLORATORY_POST_BASELINE",
        "seed": seed, "fold": fold_id, "train_n": 480, "val_n": 120,
        "pca_k": 4, "feature_map": "zz_feature_map", "feature_map_reps": 2,
        "feature_map_entanglement": "linear", "ansatz": "real_amplitudes",
        "ansatz_reps": 1, "ansatz_parameters": ansatz.num_parameters,
        "multiclass_mapping": "bitstring_modulo_3", "optimizer": "COBYLA",
        "maxiter": MAXITER, "shots": SHOTS, "initial_point": initial_point.tolist(),
        "objective_evaluations": len(history), "loss_history": history,
        "macro_f1": float(f1_score(y_val, pred, labels=CLASSES, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_val, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_val, pred)),
        "recall_high": float(recalls[2]),
        "confusion_matrix": cm.tolist(), "class_order": CLASSES,
        "predictions": [
            {"raw_row_index": int(labels.iloc[pos]["raw_row_index"]), "actual": str(a), "predicted": str(p)}
            for pos, a, p in zip(val_pos, y_val, pred)
        ],
        "fit_seconds": float(fit_seconds), "predict_seconds": float(predict_seconds),
        "feature_policy_sha256": sha256_file(ROOT / "configs/feature_policy.yaml"),
        "split_manifest_sha256": sha256_file(manifest_path),
        "final_test_used": False,
    }
    payload = json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False)
    OUT.mkdir(parents=True, exist_ok=True)
    with out_path.open("x", encoding="utf-8") as f:
        f.write(payload)
    print(f"[PASS] seed={seed}, fold={fold_id}, F1={result['macro_f1']:.4f}, "
          f"Acc={result['accuracy']:.4f}, RecallHigh={result['recall_high']:.4f}", flush=True)
    print("[SAVED]", out_path, flush=True)
    return result

## Cell 4 — **Chỉ chạy seed 7, Fold 0 trước**

Mục đích: xác minh cấu hình và đo runtime một fold. Dừng tại đây, gửi bảng PASS để kiểm tra trước khi chạy các fold còn lại. Không cần chạy lại 5 folds seed 42.

In [4]:
fold0_seed7 = run_seed_fold(seed=7, fold_id=0)
print("\n=== VQC ROBUSTNESS PREFLIGHT ===")
print("Seed:", fold0_seed7["seed"], "Fold:", fold0_seed7["fold"])
print("Macro-F1:", round(fold0_seed7["macro_f1"], 4))
print("Accuracy:", round(fold0_seed7["accuracy"], 4))
print("Recall High:", round(fold0_seed7["recall_high"], 4))
print("Evaluations:", fold0_seed7["objective_evaluations"])
print("Fit seconds:", round(fold0_seed7["fit_seconds"], 3))
print("[PASS] Seed 7 Fold 0 diagnostic saved")


=== VQC robustness seed=7 fold=0 ===
seed=7 fold=0 eval=10 loss=1.602914
seed=7 fold=0 eval=20 loss=1.607100
seed=7 fold=0 eval=30 loss=1.599778
seed=7 fold=0 eval=40 loss=1.601137
seed=7 fold=0 eval=50 loss=1.598963
seed=7 fold=0 eval=60 loss=1.598618
[PASS] seed=7, fold=0, F1=0.3446, Acc=0.4083, RecallHigh=0.4000
[SAVED] D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\vqc_k4_robust_seed7_fold0.json

=== VQC ROBUSTNESS PREFLIGHT ===
Seed: 7 Fold: 0
Macro-F1: 0.3446
Accuracy: 0.4083
Recall High: 0.4
Evaluations: 60
Fit seconds: 136.085
[PASS] Seed 7 Fold 0 diagnostic saved


## Cell 5 — Chạy **phần còn lại** sau khi Cell 4 đã PASS

Chạy các fold 1–4 seed 7, sau đó 0–4 seed 123. Dữ liệu đã lưu sẽ được xác thực và SKIP, không bị ghi đè. Không dùng seed tốt nhất để thay seed 42.

In [5]:
for seed in ROBUSTNESS_SEEDS:
    for fold_id in range(5):
        run_seed_fold(seed, fold_id)
print("[PASS] Both robustness seeds have five folds")

[SKIP] Existing verified seed=7, fold=0

=== VQC robustness seed=7 fold=1 ===
seed=7 fold=1 eval=10 loss=1.733377
seed=7 fold=1 eval=20 loss=1.614928
seed=7 fold=1 eval=30 loss=1.602364
seed=7 fold=1 eval=40 loss=1.596245
seed=7 fold=1 eval=50 loss=1.591141
seed=7 fold=1 eval=60 loss=1.589248
[PASS] seed=7, fold=1, F1=0.2843, Acc=0.3000, RecallHigh=0.3333
[SAVED] D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\vqc_k4_robust_seed7_fold1.json

=== VQC robustness seed=7 fold=2 ===
seed=7 fold=2 eval=10 loss=1.675860
seed=7 fold=2 eval=20 loss=1.657801
seed=7 fold=2 eval=30 loss=1.648671
seed=7 fold=2 eval=40 loss=1.641205
seed=7 fold=2 eval=50 loss=1.638204
seed=7 fold=2 eval=60 loss=1.634620
[PASS] seed=7, fold=2, F1=0.3583, Acc=0.3917, RecallHigh=0.4000
[SAVED] D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\vqc_k4_robust_seed7_fold2.json

=== VQC robustness seed=7 fold=3 ===
seed=7 fold=3 eval=10 loss=1.685348
seed=7 fold=3 eval=20 loss=1.621119
seed=7

## Cell 6 — QC, tổng hợp và lưu CSV (chỉ sau khi đủ 10 folds)

Đối chiếu validation IDs, recompute metrics từ predictions, khóa 5 fold mỗi seed, báo cả primary seed 42 từ JSON cũ. Không gộp hai exploratory seeds vào điểm primary; so sánh robustness chỉ là mô tả.

In [6]:
metric_rows, prediction_rows, confusion_rows = [], [], []
seed_outputs = list(ROBUSTNESS_SEEDS)
all_seeds = [PRIMARY_SEED, *seed_outputs]

for seed in all_seeds:
    for fold_id in range(5):
        path = (OUT / f"vqc_k4_fold{fold_id}_diagnostic.json") if seed == PRIMARY_SEED else (OUT / f"vqc_k4_robust_seed{seed}_fold{fold_id}.json")
        record = json.loads(path.read_text(encoding="utf-8"))
        assert record["seed"] == seed and record["fold"] == fold_id
        assert record["train_n"] == 480 and record["val_n"] == 120
        assert record["pca_k"] == 4 and record["maxiter"] == MAXITER
        assert record["shots"] == SHOTS and record["final_test_used"] is False
        assert record["class_order"] == CLASSES
        if seed != PRIMARY_SEED:
            assert record["experiment_status"] == "EXPLORATORY_POST_BASELINE"
        pf = pd.DataFrame(record["predictions"])
        assert len(pf) == 120 and pf["raw_row_index"].is_unique
        expected_ids = set(folds.loc[folds.fold_id == fold_id, "raw_row_index"].astype(int))
        assert set(pf["raw_row_index"].astype(int)) == expected_ids
        assert expected_ids.isdisjoint(test_ids)
        a = pf["actual"].astype(str).to_numpy()
        p = pf["predicted"].astype(str).to_numpy()
        for raw_id, actual in zip(pf["raw_row_index"], a):
            idx = np.flatnonzero(labels["raw_row_index"].to_numpy() == int(raw_id))
            assert len(idx) == 1 and y_array[idx[0]] == actual
        f1 = f1_score(a, p, labels=CLASSES, average="macro", zero_division=0)
        acc = accuracy_score(a, p)
        bacc = balanced_accuracy_score(a, p)
        rec = recall_score(a, p, labels=CLASSES, average=None, zero_division=0)[2]
        cm = confusion_matrix(a, p, labels=CLASSES)
        assert np.isclose(record["macro_f1"], f1)
        assert np.isclose(record["accuracy"], acc)
        assert np.isclose(record["balanced_accuracy"], bacc)
        assert np.isclose(record["recall_high"], rec)
        assert np.array_equal(record["confusion_matrix"], cm)
        metric_rows.append({
            "seed": seed, "role": "primary" if seed == PRIMARY_SEED else "exploratory_robustness",
            "fold": fold_id, "macro_f1": f1, "accuracy": acc,
            "balanced_accuracy": bacc, "recall_high": float(rec),
            "fit_seconds": record["fit_seconds"], "predict_seconds": record["predict_seconds"],
        })
        for raw_id, actual, guess in zip(pf["raw_row_index"], a, p):
            prediction_rows.append({"seed": seed, "fold": fold_id, "raw_row_index": int(raw_id), "actual": actual, "predicted": guess})
        for i, actual in enumerate(CLASSES):
            for j, guess in enumerate(CLASSES):
                confusion_rows.append({"seed": seed, "fold": fold_id, "actual": actual, "predicted": guess, "count": int(cm[i,j])})
    seed_pred = pd.DataFrame([r for r in prediction_rows if r["seed"] == seed])
    assert len(seed_pred) == 600 and seed_pred["raw_row_index"].is_unique
    overall_cm = confusion_matrix(seed_pred["actual"], seed_pred["predicted"], labels=CLASSES)
    for i, actual in enumerate(CLASSES):
        for j, guess in enumerate(CLASSES):
            confusion_rows.append({"seed": seed, "fold": "all", "actual": actual, "predicted": guess, "count": int(overall_cm[i,j])})

metrics = pd.DataFrame(metric_rows)
assert len(metrics) == 15
summary = metrics.groupby(["seed", "role"], sort=True).agg(
    folds=("fold", "count"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
    accuracy_mean=("accuracy", "mean"), balanced_accuracy_mean=("balanced_accuracy", "mean"),
    recall_high_mean=("recall_high", "mean"), fit_seconds_mean=("fit_seconds", "mean")
).reset_index()
assert summary["folds"].eq(5).all()

files = {
    "metrics": OUT / "vqc_k4_seed_robustness_metrics.csv",
    "predictions": OUT / "vqc_k4_seed_robustness_predictions.csv",
    "confusion": OUT / "vqc_k4_seed_robustness_confusion.csv",
    "summary": OUT / "vqc_k4_seed_robustness_summary.csv",
    "config": OUT / "vqc_k4_seed_robustness_config.json",
}
existing = [str(p) for p in files.values() if p.exists()]
if existing:
    raise FileExistsError(f"No overwrite permitted: {existing}")
config = {
    "experiment": "VQC_k4_seed_robustness", "status": "EXPLORATORY_POST_BASELINE",
    "primary_seed": PRIMARY_SEED, "additional_seeds": list(ROBUSTNESS_SEEDS),
    "seed_selection_note": "Two robustness seeds chosen after observing primary seed 42 results; not preregistered",
    "selection_policy": "Do not choose best seed; primary seed remains 42",
    "training_pool": "quantum_dev_pool_600", "folds": 5, "train_n": 480, "val_n": 120,
    "pca_k": 4, "features": approved,
    "preprocessing": "fit_train_only_preprocess_PCA_MinMax_clip_0_pi",
    "feature_map": "zz_feature_map(reps=2,entanglement=linear)",
    "ansatz": "real_amplitudes(reps=1,entanglement=linear)",
    "optimizer": "COBYLA", "maxiter": MAXITER,
    "shots": SHOTS, "multiclass_mapping": "bitstring_modulo_3",
    "final_test_used": False,
    "versions": {name: version(name) for name in ["qiskit", "qiskit-machine-learning", "scikit-learn", "numpy", "pandas"]},
    "split_manifest_sha256": sha256_file(manifest_path),
    "feature_policy_sha256": sha256_file(ROOT / "configs/feature_policy.yaml"),
}
metrics.to_csv(files["metrics"], index=False)
pd.DataFrame(prediction_rows).to_csv(files["predictions"], index=False)
pd.DataFrame(confusion_rows).to_csv(files["confusion"], index=False)
summary.to_csv(files["summary"], index=False)
files["config"].write_text(json.dumps(config, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
print("=== VQC K=4 SEED ROBUSTNESS ===")
print(summary.round(4).to_string(index=False))
print("[PASS] All 3 seeds, 15 fold-runs audited; 5 CSV/JSON outputs saved")

=== VQC K=4 SEED ROBUSTNESS ===
 seed                   role  folds  macro_f1_mean  macro_f1_sd  accuracy_mean  balanced_accuracy_mean  recall_high_mean  fit_seconds_mean
    7 exploratory_robustness      5         0.3147       0.0359         0.3433                  0.3370            0.3283          204.4551
   42                primary      5         0.3431       0.0335         0.3833                  0.3713            0.3250          161.4014
  123 exploratory_robustness      5         0.2971       0.0236         0.3417                  0.3157            0.2867          213.4619
[PASS] All 3 seeds, 15 fold-runs audited; 5 CSV/JSON outputs saved


## Nghiên cứu và bước kế tiếp

- **Không dùng kết quả này để thay đổi primary seed 42** hoặc tự động tuning VQC. Chỉ báo spread giữa 42/7/123 như mô tả robustness, ghi rõ exploratory bổ sung sau baseline.
- Không báo riêng fold0 của seed7 như hiệu năng 5-fold.
- Khi đủ 10 robustness runs, **Strong Classical Reference E02** là checkpoint tiếp theo trong `docs/experiment_matrix.csv`; sử dụng full development (4328) và full approved feature representation, không trộn bảng với matched k=4.
- **Không sử dụng Final Test** khi chưa hoàn tất model selection và freeze protocol.